# Catálogo de cámaras - Análisis Exploratorio de los Datos

El objetivo de este notebook es la realización de un **análisis exploratorio de los datos (EDA)** del catálogo de cámaras con el objetivo de conocer:

- El número de cámaras existentes
- La ubicación de las cámaras
- La información técnica asociada a las cámaras
- La cobertura que ofrecen para cada distrito

## Cuestiones previas

Se ha seleccionado el **Ayuntamiento de Málaga** como ámbito de aplicación.

Los datos de sus cámaras de tráfico son públicos. No expone una API pública ni endpoints estables para descarga masiva de vídeo. Por este motivo, se ha realizado una extracción de las imágenes mediante un script de cara a trabajar con ellas localmente.

Como parte del conjunto de datos, se incluyen **dos ficheros complementarios** que se describen a continuación:

- Fichero da_camarasTrafico-4326 en formato CSV: contiene los datos relevantes de cada cámara de tráfico de la ciudad. Entre otros, destacan:
    - ID y NOMBRE: identificador y nombre de la cámara.
    - DESCRIPCION y DIRECCION: texto descriptivo y dirección postal de la cámara.
    - SDOGEOMETRIA: coordenadas de la cámara en formato `POINT (lon lat)` (EPSG:4326).
    - URLIMAGEN: enlace a la imagen en tiempo real.

- Fichero da_camarasTrafico-4326 en formato GEOJSON: datos similares al dataset anterior, pero orientado a las propiedades y geometría espacial de las cámaras.

## Librerías y configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from catalogo import load_catalog
from config import path

CSV_FILE = path("catalogo")
RANDOM_SEED = 42
pd.set_option("display.max_columns", None)
print(f"Archivo CSV: {CSV_FILE}")

## Carga de datos

Se carga el dataset principal y se validan dimensiones y tipos. `load_catalog` deriva `lat`/`lon` de `SDOGEOMETRIA` (el CSV no trae esas columnas) y limpia los espacios y tabuladores de `URLIMAGEN`.

In [ ]:
if not CSV_FILE.exists():
    raise FileNotFoundError(f"No existe el archivo: {CSV_FILE}")

df = load_catalog(CSV_FILE)
print("Registros:", len(df))
print("Columnas:", len(df.columns))
print("Cámaras sin coordenadas:", df[["lat", "lon"]].isna().any(axis=1).sum())

display(df.head())
df.info()

## Información estadística

In [ ]:
nulls = df.isnull().sum().sort_values(ascending=False).to_frame("nulos")
display(nulls.head(10))

summary = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "porcentaje": round(df.isnull().mean() * 100, 2),
}).sort_values("porcentaje", ascending=False)
display(summary.head(10))

df["url_imagen_valida"] = df["URLIMAGEN"].notna()
print(df["url_imagen_valida"].value_counts())

plt.figure(figsize=(8, 8))
plt.scatter(df["lon"], df["lat"], alpha=0.7)
plt.title("Distribución de cámaras")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.grid(True, alpha=0.3)
plt.show()

duplicados = df.duplicated(subset=["ID"]).sum()
print("Duplicados:", duplicados)

## Ranking de cámaras

In [ ]:
display(df[["ID", "NOMBRE", "URLIMAGEN"]].sample(10, random_state=RANDOM_SEED))

print(f"Número de cámaras: {len(df)}")
print(f"URLs imagen válidas: {df['URLIMAGEN'].notna().sum()}")
print(f"Duplicados: {duplicados}")

## Siguiente etapa

Con este análisis se ha caracterizado el catálogo de cámaras y su distribución geográfica. La siguiente etapa (`01_EDA_Calidad_Visual.ipynb`) consiste en evaluar la calidad visual de cada imagen para etiquetar qué cámaras son útiles y cuáles deben descartarse.